# High-Dimensional Linear Regression: RMD & PGMM Methods

This notebook demonstrates the usage of:
- **RMD Lasso** Regularized Minimum Distance Lasso 
- **RMD Lasso CV** - Cross-validated penalty selection
- **PGMM** - Penalized GMM
- **PGMM-CV** - Cross-validated PGMM
- **A-PGMM** - Penalized GMM with adaptive weights
- **PGMM-CV** - Cross-validated A-PGMM

We replicate the Monte Carlo design from Section C.1.1 of Bakhitov (2026).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../replication/linear_model')))

from admliv.utils.featurizers import SimpleFeaturizer
from admliv.moments.linear_model_moment import LinearModelMoment
from admliv.core.control import PGMMControl, PGMMCVControl
from utils.control import RMDControl, RMDCVControl
from admliv.core.pgmm import PGMM
from admliv.core.pgmm_cv import PGMMCV
from utils.rmd_lasso import RMDLasso
from utils.rmd_lasso_cv import RMDLassoCV

# Monte Carlo class
from mc_hd_linear import MonteCarloHDLinear, MonteCarloConfig

np.random.seed(42)
print("Imports successful!")

Imports successful!


## 1. Data Generation

We generate data according to the design:
$$Y = X'\beta_0 + \varepsilon, \quad \varepsilon \sim N(0, 1)$$

where:
- $X = (1, X_1, \ldots, X_{100})$ with $X_j \sim N(0,1)$ i.i.d.
- $\beta_0 = (1, 1, 1, 0, \ldots, 0)$ (only 3 non-zero coefficients)
- $n = 100$ observations, $p = 101$ parameters

In [ ]:
def generate_data(n_obs=1000, p_dim=100, n_nonzero=3, seed=42):
    """Generate high-dimensional linear regression data."""
    np.random.seed(seed)
    
    # True coefficients: sparse with first n_nonzero = 1
    p_total = p_dim + 1  # including intercept
    beta_0 = np.zeros(p_total)
    beta_0[:n_nonzero] = 1.0
    
    # Generate X and Y
    X = np.random.normal(0, 1, size=(n_obs, p_dim))
    X_with_intercept = np.c_[np.ones(n_obs), X]
    eps = np.random.normal(0, 1, size=n_obs)
    Y = X_with_intercept @ beta_0 + eps
    
    # For RMD/PGMM interface
    W = {'Y': Y, 'X': X, 'Z': X}  # Z = X for exogenous case
    
    return W, beta_0

# Generate data
W, beta_0 = generate_data(n_obs=1000, p_dim=100, seed=42)

print(f"Sample size: n = {len(W['Y'])}")
print(f"Number of covariates: p = {W['X'].shape[1]} (+ intercept = {W['X'].shape[1] + 1})")
print(f"True non-zero coefficients: {np.count_nonzero(beta_0)}")
print(f"True beta_0[:5] = {beta_0[:5]}")

Sample size: n = 10000
Number of covariates: p = 100 (+ intercept = 101)
True non-zero coefficients: 3
True beta_0[:5] = [1. 1. 1. 0. 0.]


## 2. RMD Lasso (Single Penalty)

The RMD Lasso minimizes:
$$\min_\rho \frac{1}{2} \rho' G \rho - \rho' M + r_L \|D \rho\|_1$$

where $G = B'B/n$, $M = \text{mean}(m(W, b))$, and $D$ is iteratively normalized.

In [56]:
# Setup
featurizer = SimpleFeaturizer(include_bias=True)
moment = LinearModelMoment()

# RMD control parameters (CNS defaults)
control = RMDControl(
    c=0.5,        # Penalty scaling
    alpha=0.1,    # Significance level
    D_add=0.2     # Normalization stabilization
)

# Fit RMD Lasso
rmd = RMDLasso(x_featurizer=featurizer, control=control, verbose=True)
rmd.fit(W, moment)

RMD Lasso Estimation (CNS method)
n = 10000, p = 101
Penalty lambda = 0.016467
Non-zero coefficients: 6


,x_featurizer,SimpleFeaturizer()
,control,RMDControl(ma...im_divisor=40)
,verbose,True
,include_bias,True


In [57]:
# Evaluate results
rho_rmd = rmd.get_rho()

mse = np.sum((rho_rmd - beta_0) ** 2)
r2 = 1 - mse / np.sum(beta_0 ** 2)

print(f"\nRMD Lasso Results:")
print(f"  MSE: {mse:.4f}")
print(f"  R²:  {r2:.4f}")
print(f"  Non-zero coefficients: {np.count_nonzero(rho_rmd)}")
print(f"\n  Estimated rho[:5]: {rho_rmd[:5]}")
print(f"  True beta_0[:5]:   {beta_0[:5]}")


RMD Lasso Results:
  MSE: 0.0013
  R²:  0.9996
  Non-zero coefficients: 6

  Estimated rho[:5]: [1.01843819 0.97721732 0.97843723 0.         0.        ]
  True beta_0[:5]:   [1. 1. 1. 0. 0.]


## 3. RMD Lasso with Cross-Validation

Cross-validation selects the penalty parameter $c$ from a grid using the criterion:
$$CV(c) = \sum_l \sum_{i \in I_l} \left[ -2 m(W_i, b)' \rho_l(c) + (b(X_i)' \rho_l(c))^2 \right]$$

In [ ]:
# RMD with CV
featurizer_cv = SimpleFeaturizer(include_bias=True)

cv_control = RMDCVControl(
    c=0.5,
    alpha=0.1,
    D_add=0.2,
    n_folds=5,
    c_vec=[1.25, 1.0, 0.75, 0.5, 0.25]  # Grid of c values
)

rmd_cv = RMDLassoCV(x_featurizer=featurizer_cv, control=cv_control, verbose=True)
rmd_cv.fit(W, moment)

RMD Lasso CV (CNS method)
n = 10000
c values: [1.25, 1.0, 0.75, 0.5, 0.25, 0.125]
K-fold: 5

Fold 1/5

Fold 2/5

Fold 3/5

Fold 4/5

Fold 5/5

----------------------------------------
CV Results:
  c = 1.2500: CV = -6022.5772 (std = 203.4959)
  c = 1.0000: CV = -6026.8907 (std = 204.1828)
  c = 0.7500: CV = -6030.2199 (std = 204.8331)
  c = 0.5000: CV = -6031.8918 (std = 204.9924) <-- BEST
  c = 0.2500: CV = -6029.2694 (std = 205.2967)
  c = 0.1250: CV = -6021.2644 (std = 207.1367)
----------------------------------------

Refitting on full data with c = 0.5000
Non-zero coefficients: 6


,x_featurizer,SimpleFeaturizer()
,control,"RMDCVControl(...], refit=True)"
,verbose,True
,include_bias,True


In [47]:
# Evaluate RMD-CV results
rho_rmd_cv = rmd_cv.get_rho()

mse_cv = np.sum((rho_rmd_cv - beta_0) ** 2)
r2_cv = 1 - mse_cv / np.sum(beta_0 ** 2)

print(f"\nRMD Lasso CV Results:")
print(f"  Best c: {rmd_cv.best_c_}")
print(f"  MSE:    {mse_cv:.4f}")
print(f"  R²:     {r2_cv:.4f}")
print(f"  Non-zero coefficients: {np.count_nonzero(rho_rmd_cv)}")


RMD Lasso CV Results:
  Best c: 0.5
  MSE:    0.0013
  R²:     0.9996
  Non-zero coefficients: 6


## 4. PGMM (Penalized GMM)

PGMM uses the GMM objective with L1 penalty:
$$\min_\rho \frac{1}{2} \psi' \Omega \psi + \lambda \|\rho\|_1$$

For the exogenous case (Z = X), PGMM can also be used.

In [52]:
# PGMM setup
x_feat = SimpleFeaturizer(include_bias=True)
z_feat = SimpleFeaturizer(include_bias=True)
moment = LinearModelMoment()

pgmm_control = PGMMControl(c=0.01)

pgmm = PGMM(
    x_featurizer=x_feat,
    z_featurizer=z_feat,
    adaptive=False,  # set to True for A-PGMM
    # adaptive=True,
    control=pgmm_control,
    verbose=True
)
pgmm.fit(W, moment)

PGMM Two-Stage Estimation
Adaptive weights: False
Lambda: 0.000215
------------------------------------------------------------
Stage 1: Preliminary PGMM with Omega = I
  Coordinate descent converged in 4 iterations
  Final active set size: 4
  Final convergence criterion: 8.96e-06
  Preliminary estimate: 4 non-zero coefficients
------------------------------------------------------------
Stage 2: PGMM with optimal Omega
  Coordinate descent converged in 3 iterations
  Final active set size: 4
  Final convergence criterion: 2.96e-06
  Final estimate: 4 non-zero coefficients


,x_featurizer,SimpleFeaturizer()
,z_featurizer,SimpleFeaturizer()
,lambda_,np.float64(0....2831556480768)
,adaptive,False
,Omega,None
,control,PGMMControl(m...er_factor=1.1)
,verbose,True
,include_bias,True
,include_bias,True


In [53]:
# Evaluate PGMM results
rho_pgmm = pgmm.get_rho()

mse_pgmm = np.sum((rho_pgmm - beta_0) ** 2)
r2_pgmm = 1 - mse_pgmm / np.sum(beta_0 ** 2)

print(f"\nPGMM Results:")
print(f"  MSE: {mse_pgmm:.4f}")
print(f"  R²:  {r2_pgmm:.4f}")
print(f"  Non-zero coefficients: {np.count_nonzero(rho_pgmm)}")


PGMM Results:
  MSE: 0.0014
  R²:  0.9995
  Non-zero coefficients: 4


## 5. PGMM with Cross-Validation

PGMM-CV selects the penalty parameter $c$ using K-fold cross-validation with the GMM criterion:

$$CV(c) = \frac{1}{K} \sum_{k=1}^{K} \frac{1}{2} \bar{\psi}_k' \Omega_k \bar{\psi}_k$$

where:
- $\bar{\psi}_k = \frac{1}{n_k} \sum_{i \in I_k} \psi(W_i, \hat{\rho}_{\neg k}(c))$ is the mean orthogonal moment on validation fold $k$
- $\hat{\rho}_{\neg k}(c)$ is estimated on training data (excluding fold $k$)
- $\Omega_k$ is the optimal weight matrix computed from validation fold $k$

In [ ]:
# PGMM-CV setup
x_feat_cv = SimpleFeaturizer(include_bias=True)
z_feat_cv = SimpleFeaturizer(include_bias=True)

pgmm_cv_control = PGMMCVControl(
    n_folds=5,
    c_vec=np.array([0.005, 0.0075, 0.01, 0.0125, 0.015])
)

pgmm_cv = PGMMCV(
    x_featurizer=x_feat_cv,
    z_featurizer=z_feat_cv,
    adaptive=False,  # set to True for A-PGMM-CV
    # adaptive=True,
    control=pgmm_cv_control,
    verbose=True
)
pgmm_cv.fit(W, moment)

Cross-Validated PGMM Estimation
Number of folds: 5
Penalty grid size: 9
c values: [0.001  0.0025 0.005  0.0075 0.01   0.0125 0.015  0.0175 0.02  ]
  c = 0.001: 0.000292 (+/- 0.000021)
  c = 0.003: 0.000273 (+/- 0.000016)
  c = 0.005: 0.000256 (+/- 0.000014)
  c = 0.007: 0.000249 (+/- 0.000013)
  c = 0.010: 0.000248 (+/- 0.000012)
  c = 0.013: 0.000250 (+/- 0.000012)
  c = 0.015: 0.000254 (+/- 0.000013)
  c = 0.018: 0.000258 (+/- 0.000014)
  c = 0.020: 0.000264 (+/- 0.000015)

Cross-Validation Results:
----------------------------------------------------------------------
c =  0.001: CV score =   0.000292 (+/- 0.000021)
c =  0.003: CV score =   0.000273 (+/- 0.000016)
c =  0.005: CV score =   0.000256 (+/- 0.000014)
c =  0.007: CV score =   0.000249 (+/- 0.000013)
c =  0.010: CV score =   0.000248 (+/- 0.000012) <-- BEST
c =  0.013: CV score =   0.000250 (+/- 0.000012)
c =  0.015: CV score =   0.000254 (+/- 0.000013)
c =  0.018: CV score =   0.000258 (+/- 0.000014)
c =  0.020: CV score 

,x_featurizer,SimpleFeaturizer()
,z_featurizer,SimpleFeaturizer()
,adaptive,False
,Omega,None
,control,PGMMCVControl...ndom_state=42)
,verbose,True
,refit,True
,include_bias,True
,include_bias,True


In [55]:
# Evaluate PGMM-CV results
rho_pgmm_cv = pgmm_cv.get_rho()

mse_pgmm_cv = np.sum((rho_pgmm_cv - beta_0) ** 2)
r2_pgmm_cv = 1 - mse_pgmm_cv / np.sum(beta_0 ** 2)

print(f"\nPGMM-CV Results:")
print(f"  Best c: {pgmm_cv.best_c_}")
print(f"  MSE:    {mse_pgmm_cv:.4f}")
print(f"  R²:     {r2_pgmm_cv:.4f}")
print(f"  Non-zero coefficients: {np.count_nonzero(rho_pgmm_cv)}")


PGMM-CV Results:
  Best c: 0.01
  MSE:    0.0014
  R²:     0.9995
  Non-zero coefficients: 4


## 6. Comparison Summary

In [ ]:
# Create comparison table
results = pd.DataFrame({
    'Method': ['RMD', 'RMD-CV', 'PGMM', 'PGMM-CV'],
    'MSE': [mse, mse_cv, mse_pgmm, mse_pgmm_cv],
    'R²': [r2, r2_cv, r2_pgmm, r2_pgmm_cv],
    'Non-zero': [
        np.count_nonzero(rho_rmd),
        np.count_nonzero(rho_rmd_cv),
        np.count_nonzero(rho_pgmm),
        np.count_nonzero(rho_pgmm_cv)
    ]
}).set_index('Method').sort_values('MSE')

print("\n" + "="*50)
print("Single Run Comparison")
print("="*50)
print(results.to_string())

In [ ]:
# Visualize coefficient estimates
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

methods = [
    ('RMD', rho_rmd),
    ('RMD-CV', rho_rmd_cv),
    ('PGMM', rho_pgmm),
    ('PGMM-CV', rho_pgmm_cv)
]

for ax, (name, rho) in zip(axes.flat, methods):
    ax.stem(range(len(beta_0)), beta_0, linefmt='b-', markerfmt='bo', basefmt='k-', label='True β₀')
    ax.stem(range(len(rho)), rho, linefmt='r-', markerfmt='r^', basefmt='k-', label=f'Estimated ρ')
    ax.set_xlabel('Coefficient Index')
    ax.set_ylabel('Value')
    ax.set_title(f'{name} (MSE={np.sum((rho - beta_0)**2):.4f})')
    ax.legend()
    ax.set_xlim(-1, 20)  # Focus on first 20 coefficients

plt.tight_layout()
plt.show()

## 7. Monte Carlo Simulation

Run a full Monte Carlo study to compare methods across multiple replications.

In [ ]:
# Configure Monte Carlo
config = MonteCarloConfig(
    n_runs=10,      # Number of replications (increase for more stable results)
    n_obs=1000,     # Sample size
    p_dim=100,      # Number of covariates
    seed=1111,      # Base random seed
    n_nonzero=3     # True non-zero coefficients
)

print(f"Monte Carlo Configuration:")
print(f"  n_runs:    {config.n_runs}")
print(f"  n_obs:     {config.n_obs}")
print(f"  p_dim:     {config.p_dim}")
print(f"  n_nonzero: {config.n_nonzero}")

In [ ]:
# Run Monte Carlo
mc = MonteCarloHDLinear(config=config, verbose=True)
results_df = mc.run()

In [ ]:
# Display summary
print("\nMonte Carlo Summary:")
print(mc.get_summary())

In [ ]:
# Visualize MSE distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

methods = ['SGD', 'LARS', 'RMD', 'RMD-CV', 'PGMM', 'PGMM-CV', 'A-PGMM', 'A-PGMM-CV']
mse_cols = [f'{m}_MSE' for m in methods]
r2_cols = [f'{m}_R2' for m in methods]

# MSE boxplot
mse_data = [results_df[col].dropna() for col in mse_cols]
axes[0].boxplot(mse_data, labels=methods)
axes[0].set_ylabel('MSE')
axes[0].set_title('MSE Distribution by Method')
axes[0].tick_params(axis='x', rotation=45)

# R² boxplot
r2_data = [results_df[col].dropna() for col in r2_cols]
axes[1].boxplot(r2_data, labels=methods)
axes[1].set_ylabel('R²')
axes[1].set_title('R² Distribution by Method')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Save results
mc.save_results('mc_results.csv')  # Saves to mc_results_hd_linear_n{n_obs}.csv

## 8. Varying Sample Size

Compare methods across different sample sizes.

In [ ]:
# Run MC for different sample sizes
sample_sizes = [100, 1000, 10000]
summaries = {}

for n in sample_sizes:
    print(f"\n{'='*70}")
    print(f"Running n = {n}")
    print(f"{'='*70}")
    
    config_n = MonteCarloConfig(
        n_runs=10,  # Quick run
        n_obs=n,
        p_dim=100,
        seed=1111
    )
    
    mc_n = MonteCarloHDLinear(config=config_n, verbose=False)
    mc_n.run()
    summaries[n] = mc_n.get_summary()
    
    print(summaries[n][['Mean MSE', 'Mean R²']])

In [ ]:
# Plot MSE vs sample size
fig, ax = plt.subplots(figsize=(10, 6))

methods = ['SGD', 'LARS', 'RMD', 'RMD-CV', 'PGMM', 'PGMM-CV', 'A-PGMM', 'A-PGMM-CV']
colors = plt.cm.tab10(np.linspace(0, 1, len(methods)))

for method, color in zip(methods, colors):
    mse_values = [summaries[n].loc[method, 'Mean MSE'] for n in sample_sizes]
    ax.plot(sample_sizes, mse_values, 'o-', label=method, color=color, linewidth=2, markersize=8)

ax.set_xlabel('Sample Size (n)', fontsize=12)
ax.set_ylabel('Mean MSE', fontsize=12)
ax.set_title('MSE vs Sample Size', fontsize=14)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xticks(sample_sizes)

plt.tight_layout()
plt.show()